In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from scipy.optimize import curve_fit
import datetime
import pandas as pd
import re
from matplotlib.pyplot import cm
import matplotlib as mpl
from dataclasses import dataclass
# from sklearn.linear_model import LinearRegression
# import statsmodels.api as sm
from matplotlib.lines import Line2D



sys.path.insert(0,"../src/")
import common.utils as util
import common.d2d as d2d
import common.run_info as run_info

# from run_selection_single_channel import RunInfo
# import WaveformProcessor
# import FitSPE

# from common.utils import vec_regex_search


In [ ]:
params = {
    # figure
    'figure.figsize': (12, 6),
    'figure.facecolor': 'white',  # make figure background white
    # axes
    'axes.labelsize': 20,
    'axes.linewidth': 2,
    
    'axes.titlesize': 20,
    'axes.grid.which': 'both',  # gridlines at major, minor or both ticks
    # errorbar
    'errorbar.capsize': 4,
    # font
    'font.size': 18,
    'font.family': 'Times New Roman',
    # color
    'image.cmap': 'viridis',
    # legend
    'savefig.bbox': 'tight',
    'legend.fontsize': 20,
    'legend.frameon': False,
    'legend.numpoints': 1,  # only one marker in legend
    # line
    'lines.linestyle': 'solid',
    'lines.linewidth': 2,
    'lines.markeredgewidth': 1,
    'lines.markersize': 8,
    # text
    'mathtext.default': 'regular',
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
    # tick
    'xtick.top': True,  # draw ticks on the top side
    'xtick.direction': 'in',
    'xtick.labelsize': 20,
    'xtick.major.size': 8,
    'xtick.major.width': 1,
    'xtick.minor.size': 4,
    'xtick.minor.visible': False,
    'xtick.minor.width': 1,

    'ytick.right': False,  # draw ticks on the right side
    'ytick.direction': 'in',
    'ytick.labelsize': 20,
    'ytick.major.size': 6,
    'ytick.major.width': 1,
    'ytick.minor.size': 3,
    'ytick.minor.visible': False,
    'ytick.minor.width': 1,
    # GRIDS
    'grid.linestyle': '--',  ## dashed
    'mathtext.default': 'regular',
}
plt.rcParams.update(params)


In [ ]:
def create_color_scheme(color_map: str, array: object, color_range=(0,1), darken=1, reverse=False):
    values = sorted(np.unique(array))
    if reverse:
        values = values[::-1]
        
    cmap = plt.get_cmap(color_map)
    color = cmap(np.linspace(color_range[0], color_range[1], len(values)))
    
    # https://stackoverflow.com/questions/37517587/how-can-i-change-the-intensity-of-a-colormap-in-matplotlib
    color[:,0:3] *= darken
    
    clr = {values[i]: color[i] for i in range(len(values))}
    return clr

In [ ]:
# before
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/gain_info_single_channel_20240920.csv"

# after changing the SPE threshold
path_GXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250608_all_gain_info_single_channel.csv"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250609_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250610_LXe_gain_info_single_channel.csv"
path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv"

df_GXe = pd.read_csv(path_GXe, 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")

df_LXe = pd.read_csv(path_LXe, 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


df_all = pd.concat([df_GXe, df_LXe], axis=0)   


In [ ]:
class GainAnalysis:
    def __init__(self, df, output_path=None):
        self.df = df
        self.info = d2d.data(df)
        self.output_path = output_path
        
        self.settings()
        self.data_selection()
        self.voltage_calibration(calibration = False)
        self.create_dataframe()
        self.calculate_breakdown_voltage(output_path=self.output_path)


    def settings(self):
        self.color_temperature = create_color_scheme(
            "coolwarm_r", 
            self.info.temperature_K,
            darken = 0.8,
            color_range=(0.2,0.9),
            reverse=True)
        
        self.color_channel = create_color_scheme("viridis", 
                                    self.info.channel)

        self.color_voltage = create_color_scheme("hot", 
                                    self.info.voltage_preamp1_V,
                                    color_range=(0,0.8))

        self.dict_preamp_channel = {1: [0,1,2,3,4],
                       2: [5,6,7,8,10],
                       3: [9,11,12,13,14],
                       4: [15,16,17,18,19],
                       5: [20,21,22,23]}
        
        self.date_power_supply_changed = np.datetime64('2024-08-13')
        
    def data_selection(self):
        mask = (self.info.voltage_preamp1_V < -46) & (self.info.baseline_std_V < 0.025) & ~np.isnan(self.info.gain)
        self.info = self.info.apply_mask(mask)
        print("1: ", len(self.info))

        # remove files
        path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_170524.json"
        mask = ~(self.info.md_full_path == path)
        self.info = self.info.apply_mask(mask)

        paths = ['/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_171835.json',
            '/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_tritium/20241031_all_1_T98_all_voltages_6.0sig/meta_config_all_20241031_113258.json']

        for path in paths:
            mask = ~((self.info.md_full_path==path) & (self.info.channel == 2))
            self.info = self.info.apply_mask(mask)

    def voltage_calibration(self, calibration: bool):
        if calibration == True:

            preamp_boards = np.arange(1,6)

            info_corrected = self.info.copy()
            info_corrected.voltage_preamp1_V = info_corrected.voltage_preamp1_V.astype(dtype=float)

            mask_time = (info_corrected.date_time > self.date_power_supply_changed)

            for i, set_voltage in enumerate(df_bias_V_correction_avg["set"]):
                select_voltage = info_corrected.voltage_preamp1_V == set_voltage
                
                for preamp_board in preamp_boards:
                    
                    in_board_mask = np.zeros(len(info_corrected), dtype=bool)
                    for channel in self.dict_preamp_channel[preamp_board]:
                        in_board = info_corrected.channel == channel
                        in_board_mask = in_board_mask | in_board
                    
                    mask = select_voltage & in_board_mask & mask_time
                    
                    corrected_voltage = df_bias_V_correction_avg.iloc[i][f"meas{preamp_board}"]
                    
                    # modify the voltage according to preamp board and set voltage
                    info_corrected.voltage_preamp1_V[mask] = corrected_voltage

            # # also get the df of into_corrected
            # info_corrected_df = info_corrected.get_df()
            # info_corrected_df["breakdown_voltage_V"] = pd.Series(dtype='float')
            # info_corrected_df["over_voltage_V"] = pd.Series(dtype='float')

        else: 
            self.info_corrected = self.info.copy()

    def create_dataframe(self):

        self.info_corrected_df = self.info_corrected.get_df().copy()
        
        # sort again and reset index
        self.info_corrected_df.sort_values("date_time", inplace=True)
        self.info_corrected_df.reset_index(drop=True, inplace=True)
        self.info_corrected_df["run_id"] = self.info_corrected_df.index + int(1)

        # setup columns for results
        self.info_corrected_df["breakdown_voltage_V"] = pd.Series(dtype='float')
        self.info_corrected_df["over_voltage_V"] = pd.Series(dtype='float')
        self.info_corrected_df["junction_capacity"] = pd.Series(dtype='float')
        self.info_corrected_df["date_str"] = pd.Series(dtype='float')
        
        # cluster data by date
        self.info_corrected_df["date_str"] = self.info_corrected_df["date_time_str"].str[:7]
        # print(np.unique(self.info_corrected_df.date_str))

        # overwrite the info_corrected with the new df
        self.info_corrected = d2d.data(self.info_corrected_df)

    def calculate_breakdown_voltage(self, output_path):

        if self.info_corrected is None:
            raise ValueError("info_corrected is not set. Please run the data selection and voltage calibration first.")
    
        # select data before and after change of power supply
        mask = self.info_corrected.date_time < self.date_power_supply_changed
        masked_before = self.info_corrected.apply_mask(mask)

        mask = self.info_corrected.date_time > self.date_power_supply_changed
        masked_after = self.info_corrected.apply_mask(mask)

        selected_data = [masked_before, masked_after]

        # temperatures
        temperature = np.unique(self.info_corrected.temperature_K)

        # loop through data sets and calculate breakdown voltage based on the date_cluster
        for data_set in selected_data:
            for tempe in temperature:
                mask = data_set.temperature_K == tempe
                masked_temp = data_set.apply_mask(mask)

                for i, channel in enumerate(np.unique(masked_temp.channel)):
                    mask = masked_temp.channel == channel
                    masked_channel = masked_temp.apply_mask(mask)
                    
                    for date_dataset in np.sort(np.unique(masked_channel.date_str)):
                        mask = masked_channel.date_str == date_dataset
                        masked_date = masked_channel.apply_mask(mask)

                        voltage_list = np.abs(np.unique(masked_date.voltage_preamp1_V))
                        # n_points.append(len(voltage_list))
                        # date_str_list.append(date_dataset)
                        
                        if len(voltage_list) >= 3:
                            # print("Voltage points: ", voltage_list)

                            # fit
                            coef, res, _, _, _ = np.polyfit(masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)

                            # print("Date: ", date_dataset, " Channel: ", channel, " Temperature: ", tempe, " Coefficients: ", coef)
                            # print("Length of filtered data: " , len(masked_date))

                            breakdown_voltage = coef[1]/coef[0]
                            # print("Breakdown voltage: ", breakdown_voltage, " for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)


                            if breakdown_voltage > 0:
                                run_id_list = masked_date.run_id
                                # length += len(run_id_list)
                                

                                for id in run_id_list:
                                    path = self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "md_full_path"].values
                                    assert path in masked_date.md_full_path
                                    abs_bias_voltage = abs(self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "voltage_preamp1_V"])
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "breakdown_voltage_V"] = breakdown_voltage
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "over_voltage_V"] = abs_bias_voltage - breakdown_voltage
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "junction_capacity"] = coef[0]

                                
                            
                            
                            # print("Number of runs: ", len(run_id_list))
                            # print("Breakdown voltage: ", breakdown_voltage, " for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)

                        # else:
                        #     print("Not enough voltage points for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)

        if output_path is not None:
            self.info_corrected_df.to_csv(output_path, sep=',', index=False, mode='w')
        else:
            print("Output path is not set, not saving the results.")
    
        # print(np.unique(self.info_corrected_df.date_str))
        self.info_corrected_Vbd = d2d.data(self.info_corrected_df)

        mask = (self.info_corrected_Vbd.breakdown_voltage_V > 0)
        # mask = ~np.isnan(self.info_corrected_Vbd.gain) & (self.info_corrected_Vbd.breakdown_voltage_V > 0)
        self.info_corrected_Vbd = self.info_corrected_Vbd.apply_mask(mask)
        # print("6: ", len(self.info_corrected_Vbd))
        # print(np.unique(self.info_corrected_Vbd.date_str))
        

In [ ]:
result_all = GainAnalysis(df_all, output_path = None)
result_all_df = result_all.info_corrected_Vbd.get_df()


In [ ]:
info_corrected_Vbd = result_all.info_corrected_Vbd

### Breakdown Voltage

In [ ]:
plt.scatter(info_corrected_Vbd.date_time, info_corrected_Vbd.breakdown_voltage_V)

In [ ]:
plt.scatter(info_corrected_Vbd.temperature_K, info_corrected_Vbd.breakdown_voltage_V)

In [ ]:
mask = info_corrected_Vbd.temperature_K == 175
tmp = info_corrected_Vbd.apply_mask(mask)

plt.scatter(tmp.date_time, tmp.breakdown_voltage_V)

In [ ]:
for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)
    # plt.plot(tmp.junction_capacity/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])
    plt.scatter(tmp.date_time, tmp.junction_capacity/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])

plt.xlabel("Date")
plt.ylabel("Junction capacity (normalized)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
plt.grid()
plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')

In [ ]:
for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)

    plt.plot(tmp.junction_capacity/tmp.junction_capacity[0], label=f"Channel {channel}")
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')

In [ ]:
for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)
    plt.plot(tmp.breakdown_voltage_V/tmp.breakdown_voltage_V[0], label=f"Channel {channel}")
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')

### Plots

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))

df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/nEXO_2022_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = 1e3 * df_lit[gain_col_name[i]]
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"nEXO2022: {temperature_label}",
                    zorder=10
            )
    
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2018_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = 1e6 * df_lit[gain_col_name[i]] # baudis 2018
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"baudis2018: {temperature_label}",
                    zorder=10
            )
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2023_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = df_lit[gain_col_name[i]] # baudis 2018
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"baudis2023: {temperature_label}",
                    zorder=10
            )


temperature = np.unique(info_corrected_Vbd.temperature_K)

mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
gain_info = info_corrected_Vbd.apply_mask(mask)


for tempe in temperature:

    
    # print(temperature)
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        for j in range(len(masked_info)):
            # gain = masked_info.gain.mean()
            gain_list.append(masked_info.gain/33)
            
            over_voltage_list.append(masked_info.over_voltage_V)

        # over_voltage = np.array(over_voltage_list).mean()
        # gain = np.array(gain_list).mean()

        # if i == 0:
        
        plt.plot(over_voltage_list, gain_list, 
                "o-", 
                color=result_all.color_temperature[tempe]
                )
            
        # else:
        #     plt.plot(over_voltage_list, gain_list, 
        #             "o-", 
        #             color=color_temperature[temperature],
        #             


# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"NUXE-3: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1), ncol=2)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Gain")
plt.xlabel("Over Voltage")

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))

mask = ~np.isnan(info_corrected_Vbd.gain)
gain_info = info_corrected_Vbd.apply_mask(mask)

# temperatures
temperature = np.unique(info_corrected_Vbd.temperature_K)

# select data before and after change of power supply
mask = gain_info.date_time < result_all.date_power_supply_changed
masked_before = gain_info.apply_mask(mask)

mask = gain_info.date_time > result_all.date_power_supply_changed
masked_after = gain_info.apply_mask(mask)

selected_data = [masked_before, masked_after]


for data_set in selected_data:
    for tempe in temperature:
        mask = data_set.temperature_K == tempe
        masked_temp = data_set.apply_mask(mask)

        for i, channel in enumerate(np.unique(masked_temp.channel)):
            mask = masked_temp.channel == channel
            masked_channel = masked_temp.apply_mask(mask)
            
            for date_dataset in np.sort(np.unique(masked_channel.date_str)):
                mask = masked_channel.date_str == date_dataset
                masked_date = masked_channel.apply_mask(mask)

                voltage_list = np.abs(np.unique(masked_date.voltage_preamp1_V))
                # n_points.append(len(voltage_list))
                # date_str_list.append(date_dataset)
                
                if len(voltage_list) >= 3:
                    
                    # fit
                    coef, res, _, _, _ = np.polyfit(masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)
                    poly1d_fn = np.poly1d(coef)
            
            
                    ax.plot(voltage_list, poly1d_fn(voltage_list), '--',
                            color=result_all.color_channel[channel]) 
                    
                    # ax.errorbar(voltage_preamp1_V, gain_list, 
                    #     yerr=gain_err_list, 
                    #     label = f"{channel} \nV_bd: {-coef[1]/coef[0]:.2f}", 
                    #     fmt="o-", 
                    #     ecolor = color_channel[channel], 
                    #     capsize=3,
                    #     color=color_channel[channel]
                    #     )
                    

    
# info_corrected_Vbd = d2d.data(info_corrected_df)

# print(len(info_corrected_Vbd))

plt.legend(bbox_to_anchor = (1,1.1),title="Channels", ncol=2)
# plt.gca().invert_xaxis()
plt.title(f"Temperature at {temperature} K")
plt.ylabel("Gain")
plt.xlabel("Bias Voltage")

In [ ]:
temperature = np.unique(info_corrected_Vbd.temperature_K)

for tempe in temperature:

    mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
    gain_info = info_corrected_Vbd.apply_mask(mask)

    # print(temperature)
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        # for j in range(len(masked_info)):
        #     # gain = masked_info.gain.mean()
        #     gain_list.append(masked_info.gain/33)
            
        #     over_voltage_list.append(masked_info.over_voltage_V)
        
        # plt.plot(over_voltage_list, gain_list, 
        # plt.plot(masked_info.over_voltage_V, masked_info.gain/33,
        plt.plot(masked_info.date_time, masked_info.junction_capacity,
                "o", 
                color=result_all.color_temperature[tempe]
                )

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"NUXE-3: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1), ncol=2)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("junction capacity [V/s]")
plt.xlabel("date time")
plt.xticks(rotation=45, ha='right')


In [ ]:
temperature = np.unique(info_corrected_Vbd.temperature_K)

for tempe in temperature:

    mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
    gain_info = info_corrected_Vbd.apply_mask(mask)

    # print(temperature)
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        # for j in range(len(masked_info)):
        #     # gain = masked_info.gain.mean()
        #     gain_list.append(masked_info.gain/33)
            
        #     over_voltage_list.append(masked_info.over_voltage_V)
        
        # plt.plot(over_voltage_list, gain_list, 
        # plt.plot(masked_info.over_voltage_V, masked_info.gain/33,
        plt.plot(masked_info.date_time, masked_info.gain/33,
                "o", 
                color=result_all.color_temperature[tempe]
                )

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"NUXE-3: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1), ncol=2)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Gain")
plt.xlabel("date time")
#rotate x-axis labels
plt.xticks(rotation=45, ha='right')

In [ ]:
temperature = np.unique(info_corrected_Vbd.temperature_K)

for tempe in temperature:

    mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
    gain_info = info_corrected_Vbd.apply_mask(mask)

    # print(temperature)
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        # for j in range(len(masked_info)):
        #     # gain = masked_info.gain.mean()
        #     gain_list.append(masked_info.gain/33)
            
        #     over_voltage_list.append(masked_info.over_voltage_V)
        
        # plt.plot(over_voltage_list, gain_list, 
        # plt.plot(masked_info.over_voltage_V, masked_info.gain/33,
        plt.plot(masked_info.date_time, masked_info.over_voltage_V,
                "o", 
                color=result_all.color_temperature[tempe]
                )

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"NUXE-3: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1), ncol=2)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Over Voltage")
plt.xlabel("date time")
plt.xticks(rotation=45, ha='right')


In [ ]:
fig, ax = plt.subplots()

df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/nEXO_2022_breakdown.csv")

temperature_col_name = df_lit.columns[::2]
V_bd_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(temperature_col_name):
    temperature = df_lit[temperature_col_name[i]]
    V_bd = df_lit[V_bd_col_name[i]]
    ax.plot(temperature, 
            V_bd,
            "o-",
            label = f"nEXO2022: measurement {i}")
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2018_breakdown.csv")

ax.plot(df_lit["temperature"], 
        df_lit["V_bd"],
        "o-",
        label = f"baudis2018")

df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2023_breakdown.csv")

ax.plot(df_lit["temperature"], 
        df_lit["V_bd"],
        "o-",
        label = f"baudis2023")


mask = ~np.isnan(info_corrected_Vbd.breakdown_voltage_V) & (info_corrected_Vbd.breakdown_voltage_V>0)

V_bd_info = info_corrected_Vbd.apply_mask(mask)

temperature_list = np.unique(V_bd_info.temperature_K)
V_bd_list = []
V_bd_err_list = []
for temperature in np.unique(V_bd_info.temperature_K):
        mask = V_bd_info.temperature_K == temperature
        tmp = V_bd_info.apply_mask(mask)
        
        V_bd_avg = tmp.breakdown_voltage_V.mean()
        V_bd_err = np.std(tmp.breakdown_voltage_V)
        V_bd_list.append(V_bd_avg)
        V_bd_err_list.append(V_bd_err)
        
# ax.errorbar(temperature_list, 
#         V_bd_list,
#         yerr=V_bd_err_list,
#         fmt = "o-",
#         label = f"UCSD data")
ax.errorbar(temperature_list, V_bd_list, 
                 yerr=V_bd_err_list, 
                 label = f"UCSD data",                 
                 fmt="o", 
                 ecolor = "black", 
                 capsize=3
                 )


plt.legend(bbox_to_anchor = (1,1.1))
# plt.gca().invert_xaxis()
plt.ylabel("V_bd")
plt.xlabel("Temperature [K]")
